Imports Used and variable initialization

In [1]:
import os, shutil
import onnx
import torch
import torch.nn as nn

import brevitas.nn as qnn
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg

from brevitas.export import export_onnx_qcdq
from brevitas.export import export_qonnx
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.core.datatype import DataType
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.fold_constants import FoldConstants

model_dir = f"{os.environ['FINN_ROOT']}/notebooks/WS_Project"

Define model

In [2]:
class LogisticNet(nn.Module):
    def __init__(self):
        super(LogisticNet, self).__init__()
        self.pol1 = nn.MaxPool3d(2)
        self.fc1 = qnn.QuantLinear(2048, 4, bias=True, weight_bit_width=4)
        self.act1 = nn.Softmax(dim=1)

    def forward(self, input):
        out = self.pol1(input)
        out = torch.flatten(out, 1)
        out = self.fc1(out)
        out = self.act1(out)
        return out

Initialise model and load pre-trained weights.

In [3]:
net = LogisticNet()
net.load_state_dict(torch.load(f'{model_dir}/LogisticNet_84.pth', weights_only=True, map_location=torch.device('cpu')))

<All keys matched successfully>

Export to ONNX

In [4]:
model_filename = f"{model_dir}/LogisticNet_84.onnx"
tensor_x = torch.rand((1, 1, 32, 16, 32))
export_qonnx(net, (tensor_x,), model_filename, input_names=["input"])

ir_version: 7
producer_name: "pytorch"
producer_version: "1.13.1"
graph {
  node {
    input: "input"
    output: "/pol1/MaxPool_output_0"
    name: "/pol1/MaxPool"
    op_type: "MaxPool"
    attribute {
      name: "ceil_mode"
      i: 0
      type: INT
    }
    attribute {
      name: "kernel_shape"
      ints: 2
      ints: 2
      ints: 2
      type: INTS
    }
    attribute {
      name: "pads"
      ints: 0
      ints: 0
      ints: 0
      ints: 0
      ints: 0
      ints: 0
      type: INTS
    }
    attribute {
      name: "strides"
      ints: 2
      ints: 2
      ints: 2
      type: INTS
    }
  }
  node {
    input: "/pol1/MaxPool_output_0"
    output: "/Flatten_output_0"
    name: "/Flatten"
    op_type: "Flatten"
    attribute {
      name: "axis"
      i: 1
      type: INT
    }
  }
  node {
    input: "/fc1/weight_quant/export_handler/Constant_1_output_0"
    input: "/fc1/weight_quant/export_handler/Constant_2_output_0"
    input: "/fc1/weight_quant/export_handler/Con

Apply transformation to the model before converting it to FINN_ONNX.

In [5]:
model_for_sim = ModelWrapper(model_filename)

model_for_sim = model_for_sim.transform(InferShapes())
model_for_sim = model_for_sim.transform(FoldConstants())
model_for_sim = model_for_sim.transform(GiveUniqueNodeNames())
model_for_sim = model_for_sim.transform(GiveReadableTensorNames())
model_for_sim = model_for_sim.transform(InferDataTypes())
model_for_sim = model_for_sim.transform(RemoveStaticGraphInputs())

verif_model_filename = model_dir + "/LogisticNet_84_verification.onnx"
model_for_sim.save(verif_model_filename)

Conversion to FINN-ONNX

In [6]:
# clean-up
qonnx_cleanup(model_filename, out_file=model_filename)

# ModelWrapper
model = ModelWrapper(model_filename)
# Setting the input datatype explicitly because it doesn't get derived from the export function
model.set_tensor_datatype(model.graph.input[0].name, DataType["FLOAT32"])
model = model.transform(ConvertQONNXtoFINN())
model.save(model_filename)

print(f"Model saved to {model_filename}")

Model saved to /home/pantunes/sandbox/finn/notebooks/WS_Project/LogisticNet_84.onnx


/home/pantunes/sandbox/finn/deps/qonnx/src/qonnx/transformation/gemm_to_matmul.py:57: UserWarning: The GemmToMatMul transformation only offers explicit support for version 9 of the Gemm node, but the ONNX version of the supplied model is 14. Thus the transformation may fail or return incomplete results.
  warnings.warn(


Estimate FINN performance.

In [7]:
%%time
estimates_output_dir = "output_estimates_only"

#Delete previous run results if exist
if os.path.exists(estimates_output_dir):
    shutil.rmtree(estimates_output_dir)
    print("Previous run results deleted!")


cfg_estimates = build.DataflowBuildConfig(
    output_dir          = estimates_output_dir,
    mvau_wwidth_max     = 80,
    target_fps          = 1000000,
    synth_clk_period_ns = 10.0,
    fpga_part           = "xczu7ev-ffvc1156-2-e",
    steps               = build_cfg.estimate_only_dataflow_steps,
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
    ]
)

build.build_dataflow_cfg(model_filename, cfg_estimates)


Traceback (most recent call last):
  File "/home/pantunes/sandbox/finn/src/finn/builder/build_dataflow.py", line 158, in build_dataflow_cfg
    model = transform_step(model, cfg)
  File "/home/pantunes/sandbox/finn/src/finn/builder/build_dataflow_steps.py", line 326, in step_streamline
    model = model.transform(InferDataLayouts())
  File "/home/pantunes/sandbox/finn/deps/qonnx/src/qonnx/core/modelwrapper.py", line 140, in transform
    (transformed_model, model_was_changed) = transformation.apply(transformed_model)
  File "/home/pantunes/sandbox/finn/deps/qonnx/src/qonnx/transformation/infer_data_layouts.py", line 139, in apply
    raise Exception(
Exception: Unknown number of dims for input, don't know
                how to annotate


Previous run results deleted!
Building dataflow accelerator from /home/pantunes/sandbox/finn/notebooks/WS_Project/LogisticNet_84.onnx
Intermediate outputs will be generated in /tmp/finn_dev_pantunes
Final outputs will be generated in output_estimates_only
Build log is at output_estimates_only/build_dataflow.log
Running step: step_qonnx_to_finn [1/10]
Running step: step_tidy_up [2/10]
Running step: step_streamline [3/10]
> /home/pantunes/sandbox/finn/deps/qonnx/src/qonnx/transformation/infer_data_layouts.py(139)apply()
    137                 model.set_tensor_layout(inp_name, DataLayout.NC)
    138             else:
--> 139                 raise Exception(
    140                     """Unknown number of dims for input, don't know
    141                 how to annotate"""



ipdb>  16384


16384


ipdb>  exit


Build failed
CPU times: user 1.06 s, sys: 490 ms, total: 1.55 s
Wall time: 4min 10s


-1